# 04 - Risk Scoring Model

- Baseline: Logistic Regression tren bien WOE
- So sanh: LightGBM / XGBoost (+ SHAP)
- Danh gia: AUC-ROC, KS, Gini
- Chon model chinh, luu artifact vao models/

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.paths import DATA_RAW, DATA_INTERIM, DATA_PROCESSED, MODELS, REPORTS_FIGURES

## Baseline: Logistic Regression + WOE

In [2]:
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from src.data.filter_vintage import assert_no_leakage

train_df = pd.read_parquet(DATA_PROCESSED / 'train.parquet')
val_df = pd.read_parquet(DATA_PROCESSED / 'val.parquet')
test_df = pd.read_parquet(DATA_PROCESSED / 'test.parquet')

WOE_FEATURES = [c for c in train_df.columns if c.endswith('_woe')]
RAW_FEATURES = [c[:-4] for c in WOE_FEATURES]  # ten bien goc (chua WOE-transform), dung cho GBM
assert_no_leakage(RAW_FEATURES)  # lop chan cuoi truoc khi train
print('WOE features:', WOE_FEATURES)

X_train, y_train = train_df[WOE_FEATURES], train_df['bad_flag']
X_val, y_val = val_df[WOE_FEATURES], val_df['bad_flag']
X_test, y_test = test_df[WOE_FEATURES], test_df['bad_flag']

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_train_auc = roc_auc_score(y_train, lr.predict_proba(X_train)[:, 1])
lr_val_auc = roc_auc_score(y_val, lr.predict_proba(X_val)[:, 1])
lr_test_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f'LR AUC - train: {lr_train_auc:.4f}, val: {lr_val_auc:.4f}, test: {lr_test_auc:.4f}')

# --- Kiem tra dau he so ---
# Quy uoc WOE cua optbinning: WoE = ln(P(x|good) / P(x|bad)), nen bin IT rui ro co WoE DUONG
# (vd revol_util < 17.65: bad rate 12.7% < 16.3% tong the -> WoE = +0.29).
# Model du bao bad_flag = 1, nen he so ky vong AM cho MOI bien: WOE cao -> rui ro thap -> P(bad) thap.
# He so DUONG la bat thuong, can kiem tra lai binning/tuong quan giua cac bien.
#
# LUU Y: ban dau cell nay ghi nguoc ("ky vong tat ca he so > 0") nen no gan co 7 he so DUNG la
# co van de va bo sot dung 1 he so THUC SU sai (revol_util +0.228). Xem sprint_1_review.md R4.
coef_table = pd.DataFrame({'feature': WOE_FEATURES, 'coef': lr.coef_[0]}).sort_values('coef')
wrong_sign = coef_table.loc[coef_table['coef'] > 0, 'feature'].tolist()
if wrong_sign:
    print(f'\nCANH BAO - {len(wrong_sign)}/{len(WOE_FEATURES)} he so DUONG (sai dau): {wrong_sign}')
    print('  -> Kiem tra binning va tuong quan voi cac bien khac truoc khi dung lam scorecard.')
else:
    print(f'\nOK - toan bo {len(WOE_FEATURES)} he so deu AM, dung quy uoc WOE, scorecard dien giai duoc.')
coef_table


WOE features: ['fico_mid_woe', 'dti_woe', 'annual_inc_woe', 'inq_last_6mths_woe', 'home_ownership_woe', 'emp_length_years_woe', 'credit_history_length_woe', 'revol_bal_woe']


LR AUC - train: 0.6497, val: 0.6421, test: 0.6516

OK - toan bo 8 he so deu AM, dung quy uoc WOE, scorecard dien giai duoc.


,feature,coef
1,dti_woe,-0.972673
0,fico_mid_woe,-0.844517
3,inq_last_6mths_woe,-0.825000
4,home_ownership_woe,-0.625832
7,revol_bal_woe,-0.597475
6,credit_history_length_woe,-0.405219
2,annual_inc_woe,-0.293735
5,emp_length_years_woe,-0.217352


## Model so sanh: LightGBM / XGBoost

In [3]:
import lightgbm as lgb

# GBM dung feature GOC (chua WOE-transform) chu khong dung ban WOE - tan dung kha nang xu ly
# phi tuyen va categorical native cua tree model, dung tinh than so sanh "scorecard truyen
# thong vs. ML hien dai" thay vi ep 2 model dung chung 1 pipeline feature engineering.
train_raw = train_df[RAW_FEATURES].copy()
val_raw = val_df[RAW_FEATURES].copy()
test_raw = test_df[RAW_FEATURES].copy()

cat_features = [c for c in RAW_FEATURES if not pd.api.types.is_numeric_dtype(train_raw[c])]
for c in cat_features:
    train_raw[c] = train_raw[c].astype('category')
    val_raw[c] = pd.Categorical(val_raw[c], categories=train_raw[c].cat.categories)
    test_raw[c] = pd.Categorical(test_raw[c], categories=train_raw[c].cat.categories)

lgb_train = lgb.Dataset(train_raw, label=y_train, categorical_feature=cat_features, free_raw_data=False)
lgb_val = lgb.Dataset(
    val_raw, label=y_val, reference=lgb_train, categorical_feature=cat_features, free_raw_data=False
)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'min_child_samples': 100,
    'verbosity': -1,
    'seed': 42,
}
gbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)

gbm_train_auc = roc_auc_score(y_train, gbm.predict(train_raw, num_iteration=gbm.best_iteration))
gbm_val_auc = roc_auc_score(y_val, gbm.predict(val_raw, num_iteration=gbm.best_iteration))
gbm_test_auc = roc_auc_score(y_test, gbm.predict(test_raw, num_iteration=gbm.best_iteration))
print(f'LightGBM AUC - train: {gbm_train_auc:.4f}, val: {gbm_val_auc:.4f}, test: {gbm_test_auc:.4f}')
print('best_iteration:', gbm.best_iteration)


C:\Users\PC\AppData\Local\Temp\ipykernel_19452\94362546.py:14: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  test_raw[c] = pd.Categorical(test_raw[c], categories=train_raw[c].cat.categories)


Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[212]	training's auc: 0.669185	valid_1's auc: 0.651552


LightGBM AUC - train: 0.6692, val: 0.6516, test: 0.6607
best_iteration: 212


In [4]:
import shap
import matplotlib.pyplot as plt

REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)

shap_sample = test_raw.sample(min(5000, len(test_raw)), random_state=42)
explainer = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(shap_sample)
if isinstance(shap_values, list):  # mot so version tra ve list [class0, class1]
    shap_values = shap_values[1]

shap.summary_plot(shap_values, shap_sample, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / 'shap_feature_importance.png', dpi=100, bbox_inches='tight')
plt.close()

shap_importance = pd.DataFrame({
    'feature': shap_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_importance


C:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature,mean_abs_shap
0,fico_mid,0.356644
1,dti,0.201715
2,inq_last_6mths,0.163077
3,home_ownership,0.147568
4,revol_bal,0.090801
5,credit_history_length,0.068046
6,emp_length_years,0.060059
7,annual_inc,0.052499


## Danh gia (AUC, KS, Gini) va lua chon model

In [5]:
def ks_statistic(y_true, y_score) -> float:
    order = pd.DataFrame({'y': y_true, 'score': y_score}).sort_values('score', ascending=False)
    cum_bad = (order['y'] == 1).cumsum() / max((order['y'] == 1).sum(), 1)
    cum_good = (order['y'] == 0).cumsum() / max((order['y'] == 0).sum(), 1)
    return float((cum_bad - cum_good).abs().max())


def gini(auc: float) -> float:
    return 2 * auc - 1


lr_test_pred = lr.predict_proba(X_test)[:, 1]
gbm_test_pred = gbm.predict(test_raw, num_iteration=gbm.best_iteration)

results = pd.DataFrame({
    'model': ['Logistic Regression + WOE', 'LightGBM'],
    'auc_test': [lr_test_auc, gbm_test_auc],
    'ks_test': [ks_statistic(y_test.values, lr_test_pred), ks_statistic(y_test.values, gbm_test_pred)],
})
results['gini_test'] = gini(results['auc_test'])
results['auc_pass_0.68'] = results['auc_test'] >= 0.68
results['ks_pass_0.25'] = results['ks_test'] >= 0.25
print(results.to_string(index=False))

# On dinh theo thoi gian trong tap test (vintage effect o muc model performance, khong chi bad rate)
test_periods = test_df['issue_d'].dt.to_period('Q')
stability = pd.DataFrame({'period': test_periods, 'y': y_test.values, 'lr_pred': lr_test_pred, 'gbm_pred': gbm_test_pred})
stability_auc = stability.groupby('period').apply(
    lambda g: pd.Series({
        'n': len(g),
        'bad_rate': g['y'].mean(),
        'lr_auc': roc_auc_score(g['y'], g['lr_pred']) if g['y'].nunique() > 1 else float('nan'),
        'gbm_auc': roc_auc_score(g['y'], g['gbm_pred']) if g['y'].nunique() > 1 else float('nan'),
    }),
    include_groups=False,
)
print('\nOn dinh AUC theo quy trong tap test:')
stability_auc


                    model  auc_test  ks_test  gini_test  auc_pass_0.68  ks_pass_0.25
Logistic Regression + WOE  0.651641 0.219656   0.303282          False         False
                 LightGBM  0.660689 0.230769   0.321377          False         False

On dinh AUC theo quy trong tap test:


,n,bad_rate,lr_auc,gbm_auc
period,,,,
2017Q1,13240.0,0.197205,0.647128,0.655001
2017Q2,34120.0,0.209408,0.657805,0.664559
2017Q3,32962.0,0.205267,0.654024,0.661677
2017Q4,24882.0,0.179166,0.637828,0.653990


## Chọn model chính & lưu artifact

In [6]:
# Quy tac chon model chinh: neu LightGBM khong vuot Logistic Regression + WOE qua 0.02 AUC,
# uu tien LR vi de giai thich hon, chuan ngach credit risk (scorecard, de audit/tuan thu).
# Neu vuot ro ret, chon LightGBM va dung SHAP (o tren) de bu dap kha nang giai thich.
AUC_GAP_THRESHOLD = 0.02
auc_gap = gbm_test_auc - lr_test_auc
primary_model_name = 'LightGBM' if auc_gap >= AUC_GAP_THRESHOLD else 'Logistic Regression + WOE'
print(f'AUC gap (LightGBM - LR) = {auc_gap:.4f} (nguong {AUC_GAP_THRESHOLD})')
print(f'-> Model chinh de xuat: {primary_model_name}')

MODELS.mkdir(parents=True, exist_ok=True)
with open(MODELS / 'logistic_regression_woe.pkl', 'wb') as f:
    pickle.dump(lr, f)
gbm.save_model(str(MODELS / 'lightgbm_model.txt'))

with open(MODELS / 'primary_model.txt', 'w') as f:
    f.write(primary_model_name)

results.to_csv(REPORTS_FIGURES / 'model_comparison.csv', index=False)
stability_auc.to_csv(REPORTS_FIGURES / 'model_stability_by_quarter.csv')
shap_importance.to_csv(REPORTS_FIGURES / 'shap_importance.csv', index=False)

print('\nDa luu: models/logistic_regression_woe.pkl, models/lightgbm_model.txt, models/primary_model.txt')
print('Da luu: reports/figures/model_comparison.csv, model_stability_by_quarter.csv, shap_importance.csv')


AUC gap (LightGBM - LR) = 0.0090 (nguong 0.02)
-> Model chinh de xuat: Logistic Regression + WOE

Da luu: models/logistic_regression_woe.pkl, models/lightgbm_model.txt, models/primary_model.txt
Da luu: reports/figures/model_comparison.csv, model_stability_by_quarter.csv, shap_importance.csv
